# 7.9 — VGG & Inception

VGG and Inception are two classic answers to the same CNN design question: how should a network combine visual evidence across scale? VGG stacks many simple $3\times3$ convolutions so depth grows the receptive field, while Inception runs several branches in parallel, aligns their spatial shapes, concatenates their channels, and uses $1\times1$ bottlenecks to keep the multi-scale design affordable.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build VGG and Inception one idea at a time. Run each cell in order and inspect the numbers and plots — receptive field growth, channel accounting, and convolution cost are all computed directly with NumPy. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, tiny images, and shape arithmetic.
import matplotlib.pyplot as plt  # visual debugging for filters, feature maps, and cost charts.
np.random.seed(0)  # reproducibility for every synthetic image and kernel.

### 1. VGG's repeated $3\times3$ filters grow the receptive field

A convolutional activation does not see the whole image at once; it sees a **receptive field**, the input patch that can affect that activation. With stride 1 and no dilation, every new $k\times k$ convolution expands the side length by $k-1$, so stacked layers have

$$r_L = 1 + L(k-1).$$

For VGG's favorite $3\times3$ filters, each layer adds 2 pixels of reach. Two layers see $5\times5$, three see $7\times7$, and so on. The important idea is that a large view can be built by composition rather than by a single large filter.

In [ ]:
layers_w = np.arange(1, 6)  # test one through five stacked layers.
k_w = 3  # VGG-style spatial kernel size.
rf_w = 1 + layers_w * (k_w - 1)  # receptive-field side length after each layer.
print("layers:", layers_w)  # inspect the stack depths.
print("receptive-field sides:", rf_w)  # 3, 5, 7, 9, 11.
assert rf_w[1] == 5 and rf_w[2] == 7  # two 3x3 layers see 5x5; three see 7x7.

▶ What you'll see: each added $3\times3$ layer increases the receptive-field side length by exactly 2.

In [ ]:
single_5_w = 5 * 5  # one 5x5 filter for one input channel and one output channel.
two_3_w = 2 * 3 * 3  # two 3x3 filters in the same one-channel intuition.
saving_w = single_5_w - two_3_w  # parameter saving in the simplified comparison.
print("one 5x5 weights:", single_5_w)  # 25.
print("two 3x3 weights:", two_3_w, "saving:", saving_w)  # 18 and 7.
assert single_5_w == 25 and two_3_w == 18 and saving_w == 7  # lesson arithmetic.

▶ What you'll see: two small filters cover the same side length as one $5\times5$ filter with fewer one-channel weights.

In [ ]:
plt.figure(figsize=(4.6, 3))  # compare reach and simple parameter count.
plt.plot(layers_w, rf_w, marker="o", color="seagreen")  # draw receptive-field side by depth.
plt.axhline(5, color="gray", linestyle="--")  # mark the 5x5 equivalent view.
plt.title("1: VGG grows view by stacking 3×3 filters")  # title the plot.
plt.xlabel("number of 3×3 layers")  # x-axis is depth.
plt.ylabel("receptive-field side length")  # y-axis is visible input side.
plt.show()  # display the plot.

▶ What you'll see: the curve rises linearly, hitting a $5\times5$ view at two layers and $7\times7$ at three.

*Why it's done this way: stacking small filters makes receptive field a controlled function of depth. The formula $1+L(k-1)$ comes from adding only the new border each convolution can reach, and the intermediate nonlinearities let the network compose edge detectors into parts instead of applying one large linear template.*

### 2. The extra nonlinearities are part of VGG's bet

Two $3\times3$ convolutions are not just a cheaper imitation of one $5\times5$ convolution. There is a nonlinearity between them, so the stack can build a feature, threshold it, and then combine the thresholded feature. We can see this with a tiny hand-written convolution and ReLU.

In [ ]:
def conv2d_valid_w(image, kernel):  # tiny valid convolution for one channel.
    h, w = image.shape  # read input shape.
    kh, kw = kernel.shape  # read kernel shape.
    out = np.zeros((h - kh + 1, w - kw + 1))  # allocate valid output.
    for i in range(out.shape[0]):  # slide vertically.
        for j in range(out.shape[1]):  # slide horizontally.
            out[i, j] = np.sum(image[i:i+kh, j:j+kw] * kernel)  # dot product on the patch.
    return out  # return the feature map.

img_w = np.zeros((7, 7))  # create a simple image.
img_w[2:5, 2:5] = 1.0  # insert a bright square.
edge_w = np.array([[-1., 0., 1.], [-1., 0., 1.], [-1., 0., 1.]])  # vertical edge detector.
smooth_w = np.ones((3, 3)) / 9  # small averaging filter.
print("image sum:", img_w.sum())  # inspect the bright region size.

▶ What you'll see: a $7\times7$ image with a $3\times3$ bright square is ready for filtering.

In [ ]:
first_w = conv2d_valid_w(img_w, edge_w)  # detect vertical contrast.
activated_w = np.maximum(first_w, 0.0)  # ReLU keeps only positive edge evidence.
second_w = conv2d_valid_w(activated_w, smooth_w)  # combine local positive edges.
print("first map shape:", first_w.shape, "second map shape:", second_w.shape)  # 5x5 then 3x3.
print("max before ReLU:", first_w.max(), "min before ReLU:", first_w.min())  # signed edge responses.
assert first_w.shape == (5, 5) and second_w.shape == (3, 3)  # two valid 3x3 layers shrink by 4 total.

▶ What you'll see: the first layer creates signed edge evidence, ReLU removes the negative side, and the second layer aggregates what remains.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(8, 2.6))  # show the pipeline.
ax[0].imshow(img_w, cmap="gray"); ax[0].set_title("input")  # original image.
ax[1].imshow(activated_w, cmap="viridis"); ax[1].set_title("ReLU(edge)")  # thresholded edge feature.
ax[2].imshow(second_w, cmap="viridis"); ax[2].set_title("next 3×3")  # composed feature.
for a in ax: a.set_xticks([]); a.set_yticks([])  # remove tick clutter.
plt.suptitle("2: depth composes features, not just views"); plt.show()  # render.

▶ What you'll see: the intermediate feature map is sparse and directional; the next filter combines those activated edge pieces.

*Why it's done this way: without the middle ReLU, two linear convolutions collapse into one larger linear convolution. VGG's stack matters because each layer can transform and threshold the representation before the next layer mixes it, increasing expressiveness while the receptive field grows.*

### 3. Inception preserves several scales by concatenating channels

Inception makes a different architectural bet: run multiple branches on the same spatial grid — often $1\times1$, $3\times3$, $5\times5$, and pooling — then concatenate the resulting feature maps along the **channel** axis. The spatial dimensions must match, but channel counts simply add:

$$C_{out}=C_1+C_3+C_5+C_p.$$

In [ ]:
H_w, W_w = 28, 28  # a typical intermediate spatial grid.
channels_w = np.array([16, 32, 8, 8])  # branch outputs: 1x1, 3x3, 5x5, pooling projection.
labels_w = ["1×1", "3×3", "5×5", "pool"]  # branch names.
total_c_w = int(channels_w.sum())  # concatenation stacks channels.
print("branch channels:", dict(zip(labels_w, channels_w)))  # inspect branch widths.
print("output shape:", (H_w, W_w, total_c_w))  # spatial size stays fixed, channels add.
assert total_c_w == 64  # 16 + 32 + 8 + 8.

▶ What you'll see: four branches with matching $28\times28$ grids become one $28\times28\times64$ tensor.

In [ ]:
branches_w = [np.full((H_w, W_w, c), fill_value=i + 1.0) for i, c in enumerate(channels_w)]  # toy branch outputs.
concat_w = np.concatenate(branches_w, axis=2)  # Inception's assembly rule.
print("concatenated tensor shape:", concat_w.shape)  # verify channel stacking.
print("first channel values:", concat_w[0, 0, [0, 16, 48, 56]])  # sample starts of each branch block.
assert concat_w.shape == (28, 28, 64)  # shape arithmetic check.

▶ What you'll see: branch feature maps remain side by side as channel slices instead of being averaged away.

In [ ]:
plt.figure(figsize=(4.8, 3))  # visualize channel accounting.
plt.bar(labels_w, channels_w, color=["teal", "orange", "purple", "gray"])  # one bar per branch.
plt.ylabel("output channels")  # channel-count axis.
plt.title("3: Inception concatenates branch channels")  # title.
plt.show()  # display.

▶ What you'll see: the total width is the sum of branch widths; no branch has to win immediately.

*Why it's done this way: different object parts live at different scales, so concatenation keeps all scale-specific evidence available to the next layer. Adding channels is the correct operation because each branch produces a different feature basis over the same pixels.*

### 4. $1\times1$ bottlenecks make expensive branches affordable

A $5\times5$ branch is costly when it reads many input channels. Inception often first applies a $1\times1$ convolution to reduce channels, then applies the large spatial filter. Per output pixel, a convolution costs

$$k_h k_w C_{in} C_{out}.$$

The bottleneck adds a cheap projection but greatly reduces the input width of the expensive $5\times5$ filter.

In [ ]:
cin_w, reduce_w, cout_w = 64, 16, 32  # input channels, bottleneck channels, final branch channels.
direct_w = 5 * 5 * cin_w * cout_w  # direct 5x5 branch multiplies per spatial location.
reduce_cost_w = 1 * 1 * cin_w * reduce_w  # 1x1 reduction cost.
reduced_5_w = 5 * 5 * reduce_w * cout_w  # 5x5 after bottleneck.
bottle_w = reduce_cost_w + reduced_5_w  # total bottleneck branch cost.
print("direct 5x5 cost:", direct_w)  # 51200.
print("bottleneck cost:", bottle_w, "=", reduce_cost_w, "+", reduced_5_w)  # 13824.
assert direct_w == 51200 and bottle_w == 13824  # lesson arithmetic.

▶ What you'll see: reducing $64$ channels to $16$ before the $5\times5$ cuts the cost from 51200 to 13824 multiplies per location.

In [ ]:
saving_frac_w = 1 - bottle_w / direct_w  # fractional saving from the bottleneck.
print("cost reduction:", round(100 * saving_frac_w, 1), "%")  # about 73%.
plt.figure(figsize=(4.4, 3))  # compare branch costs.
plt.bar(["direct 5×5", "1×1 + 5×5"], [direct_w, bottle_w], color=["indianred", "seagreen"])  # cost bars.
plt.ylabel("multiplies per location")  # cost axis.
plt.title("4: bottleneck keeps a large branch affordable")  # title.
plt.show()  # display.

▶ What you'll see: the bottleneck bar is much shorter even though it still ends with 32 output channels.

*Why it's done this way: convolution cost multiplies by input channels and output channels. A $1\times1$ layer is a learned channel mixer, so it can compress redundant channel information before spatial filtering, lowering compute while still letting the branch learn useful combinations.*

### 5. Branches must match height and width before concatenation

Inception looks flexible, but concatenation has a strict shape rule: only the channel axis may differ. For a convolution with input size $n$, padding $p$, kernel $k$, and stride $s=1$, the output side is

$$\left\lfloor\frac{n+2p-k}{s}\right\rfloor+1.$$

A padded $3\times3$ branch on $28\times28$ stays $28\times28$; an unpadded one shrinks to $26\times26$ and cannot be concatenated with the others.

In [ ]:
def out_side_w(n, k, p=0, s=1):  # one-dimensional convolution output-size formula.
    return (n + 2 * p - k) // s + 1  # floor division matches the formula.

n_w = 28  # input side length.
same3_w = out_side_w(n_w, 3, p=1)  # padded 3x3 branch.
valid3_w = out_side_w(n_w, 3, p=0)  # unpadded 3x3 branch.
same5_w = out_side_w(n_w, 5, p=2)  # padded 5x5 branch.
print("padded 3x3:", same3_w, "unpadded 3x3:", valid3_w, "padded 5x5:", same5_w)  # inspect sizes.
assert same3_w == 28 and valid3_w == 26 and same5_w == 28  # shape arithmetic.

▶ What you'll see: padding is what keeps the $3\times3$ and $5\times5$ branches aligned with the original grid.

In [ ]:
branch_ok_w = np.zeros((28, 28, 16))  # one correctly padded branch.
branch_bad_w = np.zeros((26, 26, 16))  # one unpadded branch.
can_concat_w = branch_ok_w.shape[:2] == branch_bad_w.shape[:2]  # check spatial agreement.
print("spatial sizes:", branch_ok_w.shape[:2], branch_bad_w.shape[:2])  # compare height and width.
print("can concatenate along channels?", can_concat_w)  # False.
assert can_concat_w is False  # mismatched spatial grids cannot be channel-concatenated.

▶ What you'll see: the channel counts are irrelevant until height and width agree.

In [ ]:
plt.figure(figsize=(4.4, 3))  # visualize shape mismatch.
plt.bar(["padded", "unpadded"], [same3_w, valid3_w], color=["seagreen", "indianred"])  # output side lengths.
plt.ylabel("output side length")  # y-axis.
plt.title("5: padding aligns Inception branches")  # title.
plt.show()  # display.

▶ What you'll see: the unpadded branch is two pixels smaller per side, so it cannot join the padded branches.

*Why it's done this way: channel concatenation assumes every channel describes the same spatial coordinates. Padding is not cosmetic; it preserves the coordinate grid so feature maps from different scales can be stacked without losing alignment.*


## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for images, filters, channel tensors, and shape arithmetic.
import matplotlib.pyplot as plt # load Matplotlib for compact visual checks of feature maps and costs.
np.random.seed(0) # make random examples reproducible.

def conv2d_same(x, kernel): # compute one-channel same-padded 2D convolution with stride 1.
    pad_h = kernel.shape[0] // 2 # symmetric padding height for odd kernels.
    pad_w = kernel.shape[1] // 2 # symmetric padding width for odd kernels.
    xp = np.pad(x, ((pad_h, pad_h), (pad_w, pad_w)), mode="constant") # zero-pad the image.
    out = np.zeros_like(x, dtype=float) # allocate output with the same spatial size.
    for i in range(x.shape[0]): # slide over rows.
        for j in range(x.shape[1]): # slide over columns.
            out[i, j] = np.sum(xp[i:i+kernel.shape[0], j:j+kernel.shape[1]] * kernel) # patch-kernel dot product.
    return out # return the same-sized feature map.

def conv2d_valid(x, kernel): # compute one-channel valid convolution.
    out = np.zeros((x.shape[0] - kernel.shape[0] + 1, x.shape[1] - kernel.shape[1] + 1), dtype=float) # allocate valid output.
    for i in range(out.shape[0]): # slide vertically.
        for j in range(out.shape[1]): # slide horizontally.
            out[i, j] = np.sum(x[i:i+kernel.shape[0], j:j+kernel.shape[1]] * kernel) # patch-kernel dot product.
    return out # return the valid feature map.

def relu(x): # define the ReLU nonlinearity used between VGG-style convolutions.
    return np.maximum(x, 0.0) # keep positive responses and zero out negatives.

def conv_params(k, cin, cout): # count weights for a dense convolution without bias.
    return k * k * cin * cout # spatial area times input channels times output channels.

def out_side(n, k, p=0, s=1): # compute convolution output side length.
    return (n + 2 * p - k) // s + 1 # floor((n+2p-k)/s)+1.

def show_feature(M, title): # compact heatmap helper for single-channel feature maps.
    plt.figure(figsize=(3.4, 3)) # create a small figure.
    plt.imshow(M, cmap="viridis") # display values as color.
    plt.colorbar(label="value") # add numeric scale.
    plt.title(title) # label the plot.
    plt.xticks([]); plt.yticks([]) # hide pixel ticks.
    plt.show() # render the figure.

## 🟢 Basics (warm-up)

### Basic 1 — Compute a VGG receptive field

**Goal.** Calculate the receptive-field side length for stacked $3\times3$ filters, because VGG gets a larger view by depth. We build it in 2 steps.

In [ ]:
L_b1 = np.array([1, 2, 3]) # choose three depths to inspect.
k_b1 = 3 # VGG's standard small kernel.
rf_b1 = 1 + L_b1 * (k_b1 - 1) # apply r_L = 1 + L(k-1).
print("depths:", L_b1) # inspect layer counts.
print("receptive fields:", rf_b1) # inspect side lengths.
assert np.all(rf_b1 == np.array([3, 5, 7])) # verify canonical VGG growth.

▶ What you'll see: one, two, and three $3\times3$ layers see $3\times3$, $5\times5$, and $7\times7$ input regions.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact line plot.
plt.plot(L_b1, rf_b1, marker="o", color="teal") # plot receptive field side by depth.
plt.title("Basic 1: stacked 3×3 receptive fields") # title the plot.
plt.xlabel("layers") # label depth axis.
plt.ylabel("side length") # label receptive field axis.
plt.show() # display the plot.

▶ What you'll see: receptive-field side length grows linearly with the number of stacked filters.

👀 Takeaway: VGG trades one large kernel for a deeper sequence of small kernels.

### Basic 2 — Compare one $5\times5$ filter with two $3\times3$ filters

**Goal.** Count simple one-channel weights, because the small-filter argument starts with arithmetic before real channel complications. We build it in 2 steps.

In [ ]:
one_big_b2 = 5 * 5 # one 5x5 kernel for one input and one output channel.
two_small_b2 = 2 * 3 * 3 # two 3x3 kernels in the simplified one-channel comparison.
print("5x5 weights:", one_big_b2) # inspect direct large-kernel count.
print("two 3x3 weights:", two_small_b2) # inspect stacked small-kernel count.
assert one_big_b2 == 25 and two_small_b2 == 18 # verify 25 versus 18.

▶ What you'll see: two $3\times3$ filters use 18 weights instead of 25 in the one-channel intuition.

In [ ]:
plt.figure(figsize=(4, 3)) # create a comparison plot.
plt.bar(["one 5×5", "two 3×3"], [one_big_b2, two_small_b2], color=["gray", "seagreen"]) # draw weight counts.
plt.title("Basic 2: small filters can be cheaper") # title the chart.
plt.ylabel("weights") # label count axis.
plt.show() # display the chart.

▶ What you'll see: the stacked-small bar is shorter than the single-large bar.

👀 Takeaway: stacked $3\times3$ filters can match a $5\times5$ view while using fewer simple weights and adding a nonlinearity.

### Basic 3 — Apply a same-padded edge filter

**Goal.** Run a tiny convolution from scratch, because CNN architecture choices only matter after we understand one filter's local computation. We build it in 2 steps.

In [ ]:
img_b3 = np.zeros((7, 7)) # create a blank image.
img_b3[:, 3:] = 1.0 # add a vertical brightness step.
edge_b3 = np.array([[-1., 0., 1.], [-1., 0., 1.], [-1., 0., 1.]]) # vertical edge detector.
feat_b3 = conv2d_same(img_b3, edge_b3) # compute same-padded convolution.
print("feature shape:", feat_b3.shape) # verify same padding keeps spatial size.
assert feat_b3.shape == img_b3.shape # output grid matches input grid.

▶ What you'll see: the feature map has the same $7\times7$ spatial shape as the input.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5.5, 2.5)) # compare input and response.
ax[0].imshow(img_b3, cmap="gray"); ax[0].set_title("input step") # show the image.
ax[1].imshow(feat_b3, cmap="viridis"); ax[1].set_title("edge response") # show filter output.
for a in ax: a.set_xticks([]); a.set_yticks([]) # hide ticks.
plt.suptitle("Basic 3: local convolution response"); plt.show() # display.

▶ What you'll see: the strongest response appears near the vertical brightness change.

👀 Takeaway: a convolution produces a spatial feature map by applying the same local template everywhere.

### Basic 4 — Insert ReLU between two convolutions

**Goal.** Show the nonlinearity in a VGG-style stack, because two linear filters alone could collapse into one larger linear filter. We build it in 3 steps.

In [ ]:
img_b4 = np.zeros((7, 7)) # create a simple image.
img_b4[2:5, 2:5] = 1.0 # add a bright square.
edge_b4 = np.array([[-1., 0., 1.], [-1., 0., 1.], [-1., 0., 1.]]) # edge filter.
smooth_b4 = np.ones((3, 3)) / 9 # averaging filter.
print("input sum:", img_b4.sum()) # inspect total brightness.

In [ ]:
first_b4 = conv2d_same(img_b4, edge_b4) # first convolution.
act_b4 = relu(first_b4) # ReLU nonlinearity.
second_b4 = conv2d_same(act_b4, smooth_b4) # second convolution after the nonlinearity.
print("negative values before ReLU:", int(np.sum(first_b4 < 0))) # inspect what ReLU removes.
assert np.all(act_b4 >= 0) # ReLU output is nonnegative.

▶ What you'll see: ReLU discards negative edge evidence before the next filter mixes features.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(8, 2.5)) # show the stack.
ax[0].imshow(first_b4, cmap="coolwarm"); ax[0].set_title("conv1") # signed first response.
ax[1].imshow(act_b4, cmap="viridis"); ax[1].set_title("ReLU") # thresholded feature.
ax[2].imshow(second_b4, cmap="viridis"); ax[2].set_title("conv2") # composed feature.
for a in ax: a.set_xticks([]); a.set_yticks([]) # remove ticks.
plt.suptitle("Basic 4: VGG depth adds nonlinear composition"); plt.show() # display.

▶ What you'll see: the middle panel keeps only positive evidence, changing what the second filter can aggregate.

👀 Takeaway: VGG's depth is expressive because nonlinearities sit between the small filters.

### Basic 5 — Count branch channels in an Inception module

**Goal.** Add branch channel counts, because Inception concatenates features rather than choosing one scale. We build it in 2 steps.

In [ ]:
channels_b5 = np.array([16, 32, 8, 8]) # branch widths for 1x1, 3x3, 5x5, and pool projection.
total_b5 = int(channels_b5.sum()) # concatenate along channels.
print("branch channels:", channels_b5) # inspect branch outputs.
print("total channels:", total_b5) # inspect concatenated width.
assert total_b5 == 64 # verify 16+32+8+8.

▶ What you'll see: branch widths add to 64 channels.

In [ ]:
plt.figure(figsize=(4, 3)) # create channel-accounting plot.
plt.bar(["1×1", "3×3", "5×5", "pool"], channels_b5, color="slateblue") # branch bars.
plt.title("Basic 5: Inception branch widths") # title.
plt.ylabel("channels") # axis label.
plt.show() # display.

▶ What you'll see: the output width is a stack of separate branch contributions.

👀 Takeaway: Inception preserves multi-scale evidence by channel concatenation.

### Basic 6 — Concatenate toy branch tensors

**Goal.** Build the actual channel stack, because the formula $C_{out}=\sum C_b$ corresponds to a concrete tensor operation. We build it in 2 steps.

In [ ]:
b1_b6 = np.ones((4, 4, 2)) * 1 # toy 1x1 branch with 2 channels.
b3_b6 = np.ones((4, 4, 3)) * 3 # toy 3x3 branch with 3 channels.
b5_b6 = np.ones((4, 4, 1)) * 5 # toy 5x5 branch with 1 channel.
out_b6 = np.concatenate([b1_b6, b3_b6, b5_b6], axis=2) # concatenate along channels.
print("output shape:", out_b6.shape) # inspect stacked shape.
assert out_b6.shape == (4, 4, 6) # 2+3+1 channels.

▶ What you'll see: spatial size remains $4\times4$ while channels become 6.

In [ ]:
print("channel markers at pixel (0,0):", out_b6[0, 0, :]) # inspect how branch slices sit side by side.
plt.figure(figsize=(4, 3)) # visualize channel values at one pixel.
plt.bar(range(out_b6.shape[2]), out_b6[0, 0, :], color="darkorange") # draw one bar per concatenated channel.
plt.title("Basic 6: concatenated channel slices") # title.
plt.xlabel("channel index") # label channel axis.
plt.show() # display.

▶ What you'll see: channels from different branches keep their own values in separate slots.

👀 Takeaway: concatenation stacks features; it does not average branches together.

### Basic 7 — Compute convolution output size

**Goal.** Use the shape formula, because Inception branches must agree spatially before their channels can be stacked. We build it in 2 steps.

In [ ]:
n_b7 = 28 # input side length.
padded_b7 = out_side(n_b7, k=3, p=1, s=1) # same-padded 3x3 branch.
valid_b7 = out_side(n_b7, k=3, p=0, s=1) # unpadded 3x3 branch.
print("padded output:", padded_b7) # 28.
print("valid output:", valid_b7) # 26.
assert padded_b7 == 28 and valid_b7 == 26 # verify the lesson numbers.

▶ What you'll see: padding preserves the $28$ side length, while no padding shrinks it to $26$.

In [ ]:
plt.figure(figsize=(4, 3)) # create shape comparison.
plt.bar(["pad=1", "pad=0"], [padded_b7, valid_b7], color=["seagreen", "crimson"]) # output sizes.
plt.title("Basic 7: padding controls branch size") # title.
plt.ylabel("output side") # axis label.
plt.show() # display.

▶ What you'll see: the unpadded branch is visibly smaller and therefore incompatible with same-sized branches.

👀 Takeaway: height and width must match before channel concatenation is legal.

### Basic 8 — Count a direct $5\times5$ branch cost

**Goal.** Calculate multiplies per location for a large branch, because Inception's efficiency problem starts with expensive spatial filters. We build it in 2 steps.

In [ ]:
cin_b8, cout_b8 = 64, 32 # input and output channels.
direct_b8 = conv_params(5, cin_b8, cout_b8) # k*k*Cin*Cout.
print("direct 5x5 cost:", direct_b8) # inspect per-location cost.
assert direct_b8 == 51200 # verify 25*64*32.

▶ What you'll see: a direct $5\times5$ branch costs 51200 multiplies per output location.

In [ ]:
plt.figure(figsize=(4, 3)) # create a single-cost chart.
plt.bar(["5×5 direct"], [direct_b8], color="indianred") # draw the direct cost.
plt.ylabel("multiplies/location") # cost axis.
plt.title("Basic 8: direct large-filter cost") # title.
plt.show() # display.

▶ What you'll see: the cost is large because spatial area, input channels, and output channels multiply.

👀 Takeaway: wide inputs make large convolution branches expensive fast.

### Basic 9 — Count a $1\times1$ bottleneck branch

**Goal.** Add the $1\times1$ reduction cost and the reduced $5\times5$ cost, because bottlenecks are Inception's main efficiency trick. We build it in 3 steps.

In [ ]:
cin_b9, mid_b9, cout_b9 = 64, 16, 32 # direct input, reduced channels, final branch width.
reduce_b9 = conv_params(1, cin_b9, mid_b9) # 1x1 reduction cost.
spatial_b9 = conv_params(5, mid_b9, cout_b9) # 5x5 after reduction.
print("1x1 reduction:", reduce_b9, "reduced 5x5:", spatial_b9) # inspect pieces.
assert reduce_b9 == 1024 and spatial_b9 == 12800 # verify lesson pieces.

In [ ]:
total_b9 = reduce_b9 + spatial_b9 # total bottleneck cost.
print("bottleneck total:", total_b9) # 13824.
assert total_b9 == 13824 # verify bottleneck total.

▶ What you'll see: the bottleneck's total is much smaller than the direct $5\times5$ cost.

In [ ]:
plt.figure(figsize=(4, 3)) # create breakdown chart.
plt.bar(["1×1", "5×5 after"], [reduce_b9, spatial_b9], color=["teal", "orange"]) # show pieces.
plt.title("Basic 9: bottleneck cost pieces") # title.
plt.ylabel("multiplies/location") # axis label.
plt.show() # display.

▶ What you'll see: most remaining cost is the reduced spatial filter, but it is far cheaper than direct filtering from 64 channels.

👀 Takeaway: $1\times1$ convolutions can cheaply reduce channel width before expensive spatial work.

### Basic 10 — Compare direct and bottleneck costs

**Goal.** Put the direct and reduced branches side by side, because Inception's design is about preserving scale diversity under a compute budget. We build it in 2 steps.

In [ ]:
direct_b10 = conv_params(5, 64, 32) # direct branch cost.
bottle_b10 = conv_params(1, 64, 16) + conv_params(5, 16, 32) # bottleneck branch cost.
ratio_b10 = bottle_b10 / direct_b10 # relative cost.
print("direct:", direct_b10, "bottleneck:", bottle_b10, "ratio:", round(ratio_b10, 3)) # inspect comparison.
assert direct_b10 == 51200 and bottle_b10 == 13824 # verify canonical counts.

▶ What you'll see: the bottleneck branch costs about 27% of the direct branch.

In [ ]:
plt.figure(figsize=(4, 3)) # create cost comparison.
plt.bar(["direct", "bottleneck"], [direct_b10, bottle_b10], color=["crimson", "seagreen"]) # plot costs.
plt.title("Basic 10: Inception efficiency") # title.
plt.ylabel("multiplies/location") # axis.
plt.show() # display.

▶ What you'll see: the bottleneck makes the $5\times5$ path affordable enough to include in a multi-branch module.

👀 Takeaway: Inception can keep multiple scales because $1\times1$ reductions control branch cost.

## 🟡 Easy

### Easy 1 — Build a tiny VGG block

**Goal.** Stack two same-padded $3\times3$ convolutions with ReLU, because this is the basic VGG pattern. We build it in 3 steps.

In [ ]:
img_e1 = np.zeros((9, 9)) # create a tiny image.
img_e1[3:6, 3:6] = 1.0 # add a bright square object.
k1_e1 = np.array([[-1., 0., 1.], [-1., 0., 1.], [-1., 0., 1.]]) # vertical edge filter.
k2_e1 = np.ones((3, 3)) / 9 # averaging filter to combine activated edges.
print("input shape:", img_e1.shape) # inspect image shape.

In [ ]:
h1_e1 = relu(conv2d_same(img_e1, k1_e1)) # first VGG-style conv plus ReLU.
h2_e1 = relu(conv2d_same(h1_e1, k2_e1)) # second VGG-style conv plus ReLU.
print("block output shape:", h2_e1.shape, "max activation:", round(float(h2_e1.max()), 3)) # inspect result.
assert h2_e1.shape == img_e1.shape # same padding preserves spatial size through the block.

▶ What you'll see: the block keeps the $9\times9$ grid while transforming pixels into a composed feature map.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(8, 2.6)) # visualize the block.
ax[0].imshow(img_e1, cmap="gray"); ax[0].set_title("input") # input.
ax[1].imshow(h1_e1, cmap="viridis"); ax[1].set_title("conv+ReLU 1") # first feature.
ax[2].imshow(h2_e1, cmap="viridis"); ax[2].set_title("conv+ReLU 2") # second feature.
for a in ax: a.set_xticks([]); a.set_yticks([]) # hide ticks.
plt.suptitle("Easy 1: tiny VGG block"); plt.show() # display.

▶ What you'll see: the second feature map is smoother and more composed than the first edge response.

👀 Takeaway: a VGG block keeps repeating small same-padded convolutions to deepen features without changing the grid.

### Easy 2 — Simulate a multi-scale Inception module

**Goal.** Run simple $1\times1$, $3\times3$, $5\times5$, and pooling-style branches, because Inception compares evidence at several spatial scales. We build it in 4 steps.

In [ ]:
img_e2 = np.zeros((11, 11)) # create a small image.
img_e2[5, :] = 1.0; img_e2[:, 5] = 1.0 # draw a cross with thin and wider context.
print("input shape:", img_e2.shape) # inspect input grid.

In [ ]:
k1_e2 = np.array([[1.]]) # pointwise identity branch.
k3_e2 = np.ones((3, 3)) / 9 # medium averaging branch.
k5_e2 = np.ones((5, 5)) / 25 # larger averaging branch.
b1_e2 = conv2d_same(img_e2, k1_e2) # 1x1 branch response.
b3_e2 = conv2d_same(img_e2, k3_e2) # 3x3 branch response.
b5_e2 = conv2d_same(img_e2, k5_e2) # 5x5 branch response.
print("branch shapes:", b1_e2.shape, b3_e2.shape, b5_e2.shape) # verify alignment.
assert b1_e2.shape == b3_e2.shape == b5_e2.shape # all can concatenate.

▶ What you'll see: all branches return the same spatial grid because they use same padding.

In [ ]:
module_e2 = np.stack([b1_e2, b3_e2, b5_e2], axis=2) # stack branch maps as channels.
print("module output shape:", module_e2.shape) # inspect H, W, channels.
assert module_e2.shape == (11, 11, 3) # three branch channels.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(8, 2.6)) # visualize branch scale differences.
for a_e2, M_e2, title_e2 in zip(ax, [b1_e2, b3_e2, b5_e2], ["1×1", "3×3 avg", "5×5 avg"]): # loop through branches.
    a_e2.imshow(M_e2, cmap="viridis"); a_e2.set_title(title_e2); a_e2.set_xticks([]); a_e2.set_yticks([]) # draw branch.
plt.suptitle("Easy 2: same input, multiple scales"); plt.show() # display.

▶ What you'll see: the $1\times1$ branch keeps sharp detail, while $3\times3$ and $5\times5$ branches spread evidence over larger neighborhoods.

👀 Takeaway: Inception gives the next layer a menu of feature scales at the same spatial coordinates.

### Easy 3 — Detect and fix a branch shape mismatch

**Goal.** Check branch spatial sizes before concatenation, because mismatched grids are a common Inception bug. We build it in 3 steps.

In [ ]:
side_e3 = 28 # input side length.
sizes_e3 = np.array([out_side(side_e3, 1, p=0), out_side(side_e3, 3, p=0), out_side(side_e3, 5, p=2)]) # one branch is wrong.
print("branch output sides:", sizes_e3) # inspect 1x1, unpadded 3x3, padded 5x5.
can_concat_e3 = np.all(sizes_e3 == sizes_e3[0]) # test whether all spatial sizes match.
print("can concatenate?", can_concat_e3) # expect False.
assert can_concat_e3 == False # unpadded 3x3 causes a mismatch.

▶ What you'll see: the unpadded $3\times3$ branch has side 26 while the others have side 28.

In [ ]:
fixed_sizes_e3 = np.array([out_side(side_e3, 1, p=0), out_side(side_e3, 3, p=1), out_side(side_e3, 5, p=2)]) # choose same-padding values.
print("fixed branch sides:", fixed_sizes_e3) # inspect corrected sizes.
assert np.all(fixed_sizes_e3 == 28) # all branches now align.

In [ ]:
plt.figure(figsize=(4.5, 3)) # create before-after shape plot.
plt.plot([1, 3, 5], sizes_e3, marker="o", label="buggy padding") # old sizes.
plt.plot([1, 3, 5], fixed_sizes_e3, marker="s", label="fixed padding") # corrected sizes.
plt.title("Easy 3: branch shape alignment") # title.
plt.xlabel("kernel size") # x-axis.
plt.ylabel("output side") # y-axis.
plt.legend() # show labels.
plt.show() # display.

▶ What you'll see: the fixed line is flat at 28, which is the required condition for concatenation.

👀 Takeaway: Inception flexibility depends on strict padding arithmetic.

### Easy 4 — Budget an Inception module's parameters

**Goal.** Sum branch parameter counts, because multi-branch modules must fit a compute and memory budget. We build it in 3 steps.

In [ ]:
cin_e4 = 64 # input channel count for all branches.
branch_out_e4 = {"1x1": 16, "3x3": 32, "5x5": 8, "pool_proj": 8} # branch output widths.
params_direct_e4 = np.array([conv_params(1, cin_e4, branch_out_e4["1x1"]), conv_params(3, cin_e4, branch_out_e4["3x3"]), conv_params(5, cin_e4, branch_out_e4["5x5"]), conv_params(1, cin_e4, branch_out_e4["pool_proj"])]) # direct branch weights.
print("direct branch params:", params_direct_e4) # inspect per-branch counts.

In [ ]:
total_direct_e4 = int(params_direct_e4.sum()) # sum the module cost.
print("total direct params:", total_direct_e4) # inspect total.
assert total_direct_e4 == 32768 # 1024 + 18432 + 12800 + 512.

▶ What you'll see: the $3\times3$ and $5\times5$ spatial branches dominate the direct parameter count.

In [ ]:
plt.figure(figsize=(5, 3)) # create branch cost chart.
plt.bar(list(branch_out_e4.keys()), params_direct_e4, color="purple") # draw parameter counts.
plt.title("Easy 4: direct Inception branch parameters") # title.
plt.ylabel("weights") # y-axis.
plt.xticks(rotation=15) # fit labels.
plt.show() # display.

▶ What you'll see: spatial branches are much taller than pointwise branches.

👀 Takeaway: branch design is channel budgeting, not just picking kernel sizes.

### Easy 5 — Add bottlenecks to the expensive branches

**Goal.** Recompute module cost with $1\times1$ reductions, because Inception uses bottlenecks to make multi-scale branches practical. We build it in 3 steps.

In [ ]:
cin_e5 = 64 # input channels.
red3_e5, out3_e5 = 16, 32 # reduce before 3x3, then output 32 channels.
red5_e5, out5_e5 = 8, 8 # reduce before 5x5, then output 8 channels.
cost3_e5 = conv_params(1, cin_e5, red3_e5) + conv_params(3, red3_e5, out3_e5) # bottlenecked 3x3 branch.
cost5_e5 = conv_params(1, cin_e5, red5_e5) + conv_params(5, red5_e5, out5_e5) # bottlenecked 5x5 branch.
print("bottleneck 3x3:", cost3_e5, "bottleneck 5x5:", cost5_e5) # inspect reduced costs.

In [ ]:
other_e5 = conv_params(1, cin_e5, 16) + conv_params(1, cin_e5, 8) # 1x1 branch plus pool projection.
total_bottle_e5 = other_e5 + cost3_e5 + cost5_e5 # total bottleneck module cost.
print("total bottleneck module params:", total_bottle_e5) # inspect total.
assert total_bottle_e5 == 9280 # verified module budget.

▶ What you'll see: bottlenecks cut the module far below the direct 32768-weight version.

In [ ]:
plt.figure(figsize=(4, 3)) # compare module totals.
plt.bar(["direct", "bottleneck"], [32768, total_bottle_e5], color=["crimson", "seagreen"]) # total weights.
plt.title("Easy 5: bottlenecked module budget") # title.
plt.ylabel("weights") # y-axis.
plt.show() # display.

▶ What you'll see: the bottlenecked module keeps the same multi-branch idea with much smaller parameter count.

👀 Takeaway: pointwise reductions are what make Inception's branching affordable.

## 🔴 Advanced

### Advanced 1 — Compare receptive field and parameter growth

**Goal.** Sweep stack depth and compare receptive-field growth with parameter growth, because VGG's depth changes both view and cost. We build it in 3 steps.

In [ ]:
layers_a1 = np.arange(1, 6) # test one to five layers.
rf_a1 = 1 + layers_a1 * (3 - 1) # receptive-field side for stacked 3x3 filters.
params_a1 = layers_a1 * conv_params(3, 32, 32) # same-width 32-channel stack cost.
print("rf sides:", rf_a1) # inspect receptive fields.
print("params:", params_a1) # inspect parameters.
assert rf_a1[-1] == 11 # five 3x3 layers see 11x11.

▶ What you'll see: receptive field grows by 2 per layer while parameters grow by one full convolution per layer.

In [ ]:
single_equiv_a1 = np.array([conv_params(int(r), 32, 32) for r in rf_a1]) # direct large kernels with same receptive-field side.
ratio_a1 = params_a1 / single_equiv_a1 # stack cost versus direct large-kernel cost.
print("stack/direct ratios:", np.round(ratio_a1, 3)) # inspect efficiency.
assert ratio_a1[1] < 1 and ratio_a1[-1] < 0.5 # stacked 3x3 is cheaper than direct 5x5 or 11x11 at fixed width.

In [ ]:
plt.figure(figsize=(5, 3)) # create cost curve.
plt.plot(rf_a1, params_a1, marker="o", label="stacked 3×3") # stacked cost.
plt.plot(rf_a1, single_equiv_a1, marker="s", label="one large kernel") # direct cost.
plt.title("Advanced 1: view versus parameters") # title.
plt.xlabel("receptive-field side") # x-axis.
plt.ylabel("weights at 32 channels") # y-axis.
plt.legend() # show labels.
plt.show() # display.

▶ What you'll see: both costs rise, but the direct large-kernel curve grows much faster as the desired view widens.

👀 Takeaway: VGG's repeated $3\times3$ design is a depth-for-efficiency tradeoff with extra nonlinear transformations.

### Advanced 2 — Sweep bottleneck width

**Goal.** Vary the reduction channels before a $5\times5$ branch, because bottleneck width trades cost against representational capacity. We build it in 3 steps.

In [ ]:
mid_a2 = np.array([4, 8, 16, 32, 64]) # candidate bottleneck widths.
cin_a2, cout_a2 = 64, 32 # input and output branch channels.
costs_a2 = conv_params(1, cin_a2, mid_a2) + conv_params(5, mid_a2, cout_a2) # vectorized bottleneck costs.
direct_a2 = conv_params(5, cin_a2, cout_a2) # direct branch cost.
print("mid channels:", mid_a2) # inspect bottleneck candidates.
print("costs:", costs_a2) # inspect costs.
assert costs_a2[2] == 13824 and direct_a2 == 51200 # verify canonical mid=16 number.

▶ What you'll see: wider bottlenecks increase cost linearly, and mid=16 reproduces the lesson's 13824 count.

In [ ]:
savings_a2 = 1 - costs_a2 / direct_a2 # fractional savings versus direct branch.
print("savings %:", np.round(100 * savings_a2, 1)) # inspect cost reductions.

In [ ]:
plt.figure(figsize=(5, 3)) # create bottleneck sweep plot.
plt.plot(mid_a2, costs_a2, marker="o", color="seagreen", label="bottleneck cost") # reduced branch cost.
plt.axhline(direct_a2, color="crimson", linestyle="--", label="direct 5×5") # direct reference.
plt.title("Advanced 2: bottleneck width tradeoff") # title.
plt.xlabel("reduction channels") # x-axis.
plt.ylabel("multiplies/location") # y-axis.
plt.legend() # show labels.
plt.show() # display.

▶ What you'll see: tiny reductions are cheapest, but a wider reduction preserves more channel information at higher cost.

👀 Takeaway: the bottleneck dimension is a capacity knob, not just an implementation detail.

### Advanced 3 — Build a valid Inception shape checker

**Goal.** Implement a small checker for branch shapes, because real architecture code must reject invalid concatenations early. We build it in 3 steps.

In [ ]:
branches_a3 = [(1, 0, 16), (3, 1, 32), (5, 2, 8), (3, 0, 8)] # (kernel, padding, channels); last branch is intentionally wrong.
input_side_a3 = 28 # common input side.
sides_a3 = np.array([out_side(input_side_a3, k_a3, p_a3) for k_a3, p_a3, c_a3 in branches_a3]) # compute spatial side per branch.
channels_a3 = np.array([c_a3 for k_a3, p_a3, c_a3 in branches_a3]) # read output channels.
print("sides:", sides_a3, "channels:", channels_a3) # inspect branch metadata.

In [ ]:
valid_a3 = np.all(sides_a3 == sides_a3[0]) # concatenate requires equal spatial sides.
print("shape check passed?", valid_a3) # expect False.
assert valid_a3 == False # the unpadded 3x3 branch gives side 26.

▶ What you'll see: the checker catches a $26\times26$ branch among $28\times28$ branches.

In [ ]:
fixed_branches_a3 = [(1, 0, 16), (3, 1, 32), (5, 2, 8), (3, 1, 8)] # fix the final branch padding.
fixed_sides_a3 = np.array([out_side(input_side_a3, k_a3, p_a3) for k_a3, p_a3, c_a3 in fixed_branches_a3]) # recompute sides.
valid_fixed_a3 = np.all(fixed_sides_a3 == fixed_sides_a3[0]) # check again.
print("fixed sides:", fixed_sides_a3, "passed?", valid_fixed_a3) # inspect fixed result.
assert valid_fixed_a3 == True and int(channels_a3.sum()) == 64 # fixed shape and total channel count.

In [ ]:
labels_a3 = [f"{k_a3}×{k_a3}, p={p_a3}" for k_a3, p_a3, c_a3 in fixed_branches_a3] # label fixed branches.
plt.figure(figsize=(5, 3)) # create branch channel plot.
plt.bar(labels_a3, channels_a3, color=["steelblue", "seagreen", "orange", "purple"]) # show output channels per branch.
plt.title(f"Advanced 3: branch channels concatenate to {int(channels_a3.sum())}") # title with total.
plt.xlabel("fixed branch") # x-axis.
plt.ylabel("output channels") # y-axis.
plt.show() # display.

▶ What you'll see: the four fixed branches contribute 16, 32, 8, and 8 channels, summing to a 64-channel concatenation.

▶ What you'll see: after padding is corrected, all branches align at side 28 and can produce 64 concatenated channels.

👀 Takeaway: Inception modules need explicit shape accounting before channel accounting matters.

### Advanced 4 — Compare VGG-style and Inception-style feature diversity

**Goal.** Apply one sequential stack and one parallel module to the same image, because the architectures organize feature evidence differently. We build it in 4 steps.

In [ ]:
img_a4 = np.zeros((13, 13)) # create a synthetic image.
img_a4[3:10, 6] = 1.0; img_a4[6, 3:10] = 1.0 # draw a cross.
img_a4[4:9, 4:9] += 0.4 # add a broader square context.
print("image max:", img_a4.max(), "shape:", img_a4.shape) # inspect the input.

In [ ]:
edge_a4 = np.array([[-1., 0., 1.], [-1., 0., 1.], [-1., 0., 1.]]) # vertical edge kernel.
smooth_a4 = np.ones((3, 3)) / 9 # local averaging kernel.
vgg_a4 = relu(conv2d_same(relu(conv2d_same(img_a4, edge_a4)), smooth_a4)) # sequential VGG-style feature.
print("VGG feature max:", round(float(vgg_a4.max()), 3)) # inspect sequential response.

In [ ]:
inc1_a4 = img_a4.copy() # 1x1 branch.
inc3_a4 = conv2d_same(img_a4, np.ones((3, 3)) / 9) # 3x3 context.
inc5_a4 = conv2d_same(img_a4, np.ones((5, 5)) / 25) # 5x5 context.
inc_a4 = np.stack([inc1_a4, inc3_a4, inc5_a4], axis=2) # parallel branch output.
print("Inception tensor shape:", inc_a4.shape) # inspect channel stack.
assert inc_a4.shape == (13, 13, 3) # three scale channels.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(10, 2.5)) # compare sequential versus parallel views.
for a_a4, M_a4, title_a4 in zip(ax, [vgg_a4, inc_a4[:, :, 0], inc_a4[:, :, 1], inc_a4[:, :, 2]], ["VGG stack", "Inc 1×1", "Inc 3×3", "Inc 5×5"]): # draw maps.
    a_a4.imshow(M_a4, cmap="viridis"); a_a4.set_title(title_a4); a_a4.set_xticks([]); a_a4.set_yticks([]) # plot one feature.
plt.suptitle("Advanced 4: sequential composition vs parallel scales"); plt.show() # display.

▶ What you'll see: the VGG stack shows one composed response, while Inception preserves sharp, medium, and broad versions as separate channels.

👀 Takeaway: VGG commits to depth-first composition; Inception delays the scale decision by preserving parallel feature channels.

### Advanced 5 — Estimate module compute over a spatial grid

**Goal.** Convert per-location costs into whole-feature-map costs, because architecture choices multiply by every output pixel. We build it in 3 steps.

In [ ]:
H_a5, W_a5 = 28, 28 # output spatial grid.
locations_a5 = H_a5 * W_a5 # number of output positions.
direct_per_a5 = conv_params(5, 64, 32) # direct per-location cost.
bottle_per_a5 = conv_params(1, 64, 16) + conv_params(5, 16, 32) # bottleneck per-location cost.
print("locations:", locations_a5) # inspect grid size.
assert locations_a5 == 784 # 28*28.

In [ ]:
direct_total_a5 = locations_a5 * direct_per_a5 # total direct branch multiplies.
bottle_total_a5 = locations_a5 * bottle_per_a5 # total bottleneck branch multiplies.
print("direct total:", direct_total_a5) # inspect whole-map cost.
print("bottleneck total:", bottle_total_a5) # inspect whole-map reduced cost.
assert direct_total_a5 == 40140800 and bottle_total_a5 == 10838016 # verified totals.

▶ What you'll see: a per-location saving becomes tens of millions of multiplies over a $28\times28$ feature map.

In [ ]:
plt.figure(figsize=(4.5, 3)) # create total-compute chart.
plt.bar(["direct", "bottleneck"], [direct_total_a5 / 1e6, bottle_total_a5 / 1e6], color=["crimson", "seagreen"]) # plot in millions.
plt.title("Advanced 5: per-location savings scale up") # title.
plt.ylabel("million multiplies") # y-axis.
plt.show() # display.

▶ What you'll see: bottlenecking saves roughly 29 million multiplies on this one branch and grid.

👀 Takeaway: Inception's $1\times1$ reductions matter because small per-pixel savings are repeated across the entire feature map.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

VGG favors simple repeated small filters, while Inception asks several filter sizes to inspect the same image in parallel.

VGG made depth feel systematic by stacking many 3 by 3 convolutions. Inception made width systematic by running branches at multiple scales and concatenating their channels. Both designs are best understood with receptive-field, channel, and cost arithmetic.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)


## The concept, built once (D1)
$$R_{\text{two }3\times3}=3+(3-1)=5,\qquad C_{concat}=\sum_b C_b$$

We first write the reusable method and assert the exact lesson numbers before scaling to the ladder.

In [ ]:

def vgg_or_inception_features():
    receptive_field = 3 + (3 - 1)
    one_5x5_weights = 5 * 5
    two_3x3_weights = 2 * 3 * 3
    branch_channels = np.array([16, 32, 8, 8])
    concat_channels = int(branch_channels.sum())
    direct_cost = 5 * 5 * 64 * 32
    bottleneck_cost = 1 * 1 * 64 * 16 + 5 * 5 * 16 * 32
    return receptive_field, one_5x5_weights, two_3x3_weights, concat_channels, direct_cost, bottleneck_cost


receptive_field, one_5x5_weights, two_3x3_weights, concat_channels, direct_cost, bottleneck_cost = vgg_or_inception_features()
padded_size = 28
unpadded_size = 26

assert receptive_field == 5
assert one_5x5_weights == 25
assert two_3x3_weights == 18
assert concat_channels == 64
assert direct_cost == 51200
assert bottleneck_cost == 13824
assert padded_size != unpadded_size

print("receptive field", receptive_field)
print("5x5 vs two 3x3 weights", one_5x5_weights, two_3x3_weights)
print("concat channels", concat_channels)
print("bottleneck cost", bottleneck_cost)


## Visual check
The numbers above are easier to trust when the intermediate feature behavior is visible.

In [ ]:

labels = ["single 5x5", "two 3x3", "direct branch", "bottleneck"]
values = [one_5x5_weights, two_3x3_weights, direct_cost, bottleneck_cost]

fig, axes = plt.subplots(1, 2, figsize=(9, 3))

axes[0].bar(labels[:2], values[:2])
axes[0].set_title("toy weights")

axes[1].bar(labels[2:], values[2:])
axes[1].set_title("branch multiplies")

for ax in axes:
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


## Dataset ladder (D1 to D5)
We inline the shared CPU-safe classification ladder. Each rung returns images `X` with shape `(n, 8, 8)` and labels `y`, so the same featurizer can be evaluated from hand patches to the hardest fallback or cached MNIST rung.

In [ ]:
"""
F6 (Vision) shared dataset ladder — D1..D5 of rising complexity, CPU-only and run-all-safe.

This is the canonical ladder inlined into the classification-style Part-7 notebooks. Every
rung returns (X, y) with X shape (n, 8, 8) float in [0, 1] and integer labels y, so one
featurizer + classifier can run unchanged across all five rungs (the "watch it scale" story).

D4/D5 load real MNIST / CIFAR-10 via torchvision when the download is available (as in Colab),
offline they fall back to a harder synthetic set so run-all never fails. Code is written one
statement per line for readability.
"""

import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


def _resize_to_8x8(img):
    """Nearest-neighbour resize of a 2-D array to 8x8 (no SciPy dependency)."""
    h, w = img.shape
    rows = (np.linspace(0, h - 1, 8)).round().astype(int)
    cols = (np.linspace(0, w - 1, 8)).round().astype(int)
    return img[np.ix_(rows, cols)]


def _normalize(x):
    """Scale an array into [0, 1], a flat array becomes all zeros."""
    x = x.astype(float)
    lo = x.min()
    hi = x.max()
    if hi - lo < 1e-12:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)


def d1_hand_patches():
    """D1 — hand-built 4x4 patches, 2 classes: a vertical line (col 1) vs a horizontal line (row 1).

    Fixed positions with light jitter, so the two classes are cleanly separable and the
    mechanism is fully visible — the easy first rung.
    """
    rng = np.random.default_rng(0)
    images = []
    labels = []
    for _ in range(24):
        patch = rng.uniform(0.0, 0.15, size=(4, 4))
        patch[:, 1] = rng.uniform(0.85, 1.0)
        images.append(_resize_to_8x8(patch))
        labels.append(0)
    for _ in range(24):
        patch = rng.uniform(0.0, 0.15, size=(4, 4))
        patch[1, :] = rng.uniform(0.85, 1.0)
        images.append(_resize_to_8x8(patch))
        labels.append(1)
    return np.array(images), np.array(labels)


def d2_synthetic_shapes():
    """D2 — clean synthetic shapes on an 8x8 grid, 2 classes (square vs disc)."""
    rng = np.random.default_rng(1)
    yy, xx = np.mgrid[0:8, 0:8]
    images = []
    labels = []
    for _ in range(80):
        img = rng.uniform(0.0, 0.1, size=(8, 8))
        img[2:6, 2:6] = 0.9
        images.append(img)
        labels.append(0)
    for _ in range(80):
        img = rng.uniform(0.0, 0.1, size=(8, 8))
        disc = (xx - 3.5) ** 2 + (yy - 3.5) ** 2 <= 4.0
        img[disc] = 0.9
        images.append(img)
        labels.append(1)
    return np.array(images), np.array(labels)


def d3_sklearn_digits():
    """D3 — real sklearn digits (native 8x8), 4 classes for a fast, honest multi-class rung."""
    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    X = digits.images[keep]
    y = digits.target[keep]
    X = np.array([_normalize(img) for img in X])
    return X, y


def _synthetic_textured(n_per_class, n_classes, noise, seed):
    """A harder synthetic fallback: textured class prototypes at 8x8 with noise."""
    rng = np.random.default_rng(seed)
    protos = [rng.uniform(0.0, 1.0, size=(8, 8)) for _ in range(n_classes)]
    images = []
    labels = []
    for cls in range(n_classes):
        for _ in range(n_per_class):
            img = protos[cls] + rng.normal(0.0, noise, size=(8, 8))
            images.append(_normalize(img))
            labels.append(cls)
    return np.array(images), np.array(labels)


def _call_with_timeout(fn, seconds):
    """Run fn() but abort with TimeoutError after `seconds` (guards slow/hanging downloads)."""
    import signal

    def _raise(signum, frame):
        raise TimeoutError("download timed out")

    old = signal.signal(signal.SIGALRM, _raise)
    signal.alarm(seconds)
    try:
        return fn()
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old)


def _load_mnist_gray(classes, n_per_class, seed, shift=False, noise=0.0):
    """Load MNIST via torchvision, grayscale + resize to 8x8, subsample. Raises on failure.

    MNIST is a small (~11 MB) real dataset. CIFAR-10 is deliberately avoided (a 170 MB
    download breaks run-all-safety), the harder D5 rung instead shifts and noises MNIST.
    """
    import torchvision

    ds = torchvision.datasets.MNIST(root="./data", train=True, download=True)
    rng = np.random.default_rng(seed)
    targets = np.array(ds.targets)
    images = []
    labels = []
    for cls in classes:
        idx = np.where(targets == cls)[0][:n_per_class]
        for i in idx:
            arr = np.asarray(ds[int(i)][0], dtype=float)
            small = _resize_to_8x8(arr)
            if shift:
                small = np.roll(small, rng.integers(-1, 2), axis=0)
                small = np.roll(small, rng.integers(-1, 2), axis=1)
            if noise:
                small = small + rng.normal(0.0, noise * 255.0, size=(8, 8))
            images.append(_normalize(small))
            labels.append(cls)
    return np.array(images), np.array(labels)


def d4_mnist_or_fallback():
    """D4 — real MNIST (4 clean classes) when downloadable, else a harder synthetic set."""
    try:
        X, y = _call_with_timeout(lambda: _load_mnist_gray([0, 1, 2, 3], 60, seed=4), 30)
        return (X, y), "MNIST (real)"
    except Exception:
        return _synthetic_textured(60, 4, noise=0.35, seed=4), "synthetic (offline fallback)"


def d5_mnist_hard_or_fallback():
    """D5 — real MNIST, more classes with shift + noise (distribution shift), else hardest synthetic."""
    try:
        X, y = _call_with_timeout(lambda: _load_mnist_gray([0, 1, 2, 3, 4, 5], 60, seed=5, shift=True, noise=0.12), 30)
        return (X, y), "MNIST shifted+noisy (real, harder)"
    except Exception:
        return _synthetic_textured(60, 6, noise=0.6, seed=5), "synthetic (offline fallback)"


def load_ladder():
    """Return the five rungs as a list of (name, X, y). D4/D5 note whether real data loaded."""
    rungs = []
    rungs.append(("D1 hand patches", *d1_hand_patches()))
    rungs.append(("D2 synthetic shapes", *d2_synthetic_shapes()))
    rungs.append(("D3 sklearn digits", *d3_sklearn_digits()))
    (x4, y4), tag4 = d4_mnist_or_fallback()
    rungs.append((f"D4 {tag4}", x4, y4))
    (x5, y5), tag5 = d5_mnist_hard_or_fallback()
    rungs.append((f"D5 {tag5}", x5, y5))
    return rungs


def accuracy_with(featurize, X, y):
    """Map each image through featurize, train logistic regression, return held-out accuracy."""
    feats = np.array([featurize(img) for img in X])
    x_tr, x_te, y_tr, y_te = train_test_split(feats, y, test_size=0.4, random_state=0, stratify=y)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.score(x_te, y_te)




rungs = load_ladder()

fig, axes = plt.subplots(1, 5, figsize=(12, 3))

for ax, (name, X, y) in zip(axes, rungs):
    ax.imshow(X[0], cmap="gray")
    ax.set_title(f"{name.split()[0]}\n{X.shape}\n{len(set(y.tolist()))} classes")
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

plt.tight_layout()
plt.show()

for name, X, y in rungs:
    print(f"{name:38s} X={X.shape} classes={sorted(set(y.tolist()))}")


## Run the same method across D1-D5
Only the data rung changes. The featurizer and accuracy metric stay fixed.

In [ ]:

def blur3(img):
    padded = np.pad(img, 1, mode="edge")
    out = np.zeros_like(img)
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            out[i, j] = padded[i:i + 3, j:j + 3].mean()
    return out


def topic_feature_map(img):
    local = blur3(img)
    detail = img - local
    branch = np.maximum(detail, 0.0)
    return local, branch


def featurize(img):
    local, branch = topic_feature_map(img)
    return np.concatenate([img.ravel(), local.ravel(), branch.ravel()])


accuracies = []

for name, X, y in rungs:
    acc = accuracy_with(featurize, X, y)
    accuracies.append(acc)
    print(f"{name:38s} accuracy={acc:.3f}")

baseline_accuracies = []

for name, X, y in rungs:
    acc = accuracy_with(lambda im: im.ravel(), X, y)
    baseline_accuracies.append(acc)

print("flat baseline", [round(x, 3) for x in baseline_accuracies])


## Results visualization
Top row: one feature or activation panel per rung. Bottom row: accuracy versus ladder complexity.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for idx, (name, X, y) in enumerate(rungs):
    feature = topic_feature_map(X[0])
    if isinstance(feature, tuple):
        feature = feature[0]
    axes[0, idx].imshow(feature, cmap="viridis")
    axes[0, idx].set_title(name.split()[0])
    axes[0, idx].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

axes[1, 0].plot(range(1, 6), accuracies, marker="o", label="topic features")
axes[1, 0].plot(range(1, 6), baseline_accuracies, marker="s", label="flat baseline")
axes[1, 0].set_xticks(range(1, 6))
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("accuracy")
axes[1, 0].set_ylim(0.0, 1.05)
axes[1, 0].legend()
axes[1, 0].set_title("accuracy vs rung")

for ax in axes[1, 1:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## Pitfall on D5: concatenating mismatched branch sizes
Inception concatenation requires every branch to agree on spatial height and width. The unpadded 3 by 3 branch shrinks 28 to 26, so we assert the mismatch and then fix it with padding.

In [ ]:

input_size = 28
same_padding_branch = input_size
valid_branch = input_size - 3 + 1
try:
    assert same_padding_branch == valid_branch
except AssertionError:
    print("concat fails", same_padding_branch, "vs", valid_branch)

fixed_valid_branch = input_size
assert same_padding_branch == fixed_valid_branch
print("fixed concat sizes", same_padding_branch, fixed_valid_branch)


## Evaluate it + Practice
- Metric: held-out accuracy on every rung, compared with a no-skill flat-pixel logistic baseline.
- Sanity check: D1 should be easy enough to overfit or nearly overfit with the concept features.
- Ablation: remove the key block feature and accuracy should not improve over the flat baseline.
- Failure signal: the hardest rung may expose distribution shift, shape mismatch, or compute-budget mistakes before D1 does.

Practice prompts:
1. Change one design constant and rerun the accuracy curve.
2. Print the D5 confusion pattern for the worst two classes.
3. Replace the feature map panel with an example from a different class.

In [ ]:
# Your code here


In [ ]:
# Your code here


In [ ]:
# Your code here
